# GARCH-Family Volatility Forecasting and Value-at-Risk Backtesting

## S&P 500 Daily Returns | 2015–2024

### Project Overview

This project develops an end-to-end volatility forecasting and market risk analysis pipeline using daily S&P 500 returns.

The analysis focuses on five main objectives:

1. Test whether S&P 500 returns are stationary.
2. Detect volatility clustering and ARCH effects.
3. Compare GARCH-family volatility models.
4. Forecast volatility and estimate 99% Value-at-Risk (VaR).
5. Evaluate the VaR forecasts using out-of-sample backtesting and the Kupiec unconditional coverage test.

### Models Considered

- GARCH(1,1)
- EGARCH(1,1)
- GJR-GARCH(1,1)

### Overall Workflow

**S&P 500 Prices**  
↓  
**Log Returns**  
↓  
**ADF Stationarity Test**  
↓  
**Squared Returns → Ljung-Box + ACF**  
↓  
**GARCH / EGARCH / GJR-GARCH**  
↓  
**AIC / BIC Model Comparison**  
↓  
**Select Preferred Model**  
↓  
**Walk-Forward Volatility Forecasting**  
↓  
**99% Value-at-Risk**  
↓  
**VaR Backtesting**  
↓  
**Kupiec Coverage Test**

In [1]:
!pip install arch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 982.9/982.9 kB 11.3 MB/s eta 0:00:00


## 1. Environment and Libraries

The analysis is implemented in Python using standard numerical, statistical, visualization, and financial time-series libraries.

The main packages used are:

- **NumPy** and **Pandas** for numerical and data manipulation tasks
- **Matplotlib** for visualization
- **Statsmodels** for statistical tests and autocorrelation analysis
- **ARCH** for GARCH-family volatility models
- **SciPy** for probability distributions and statistical tests
- **yfinance** for retrieving historical S&P 500 data

The required packages are listed in `requirements.txt` for reproducibility.

In [2]:
# importing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from arch import arch_model

### Reproducibility

A fixed random seed is set to make any operations involving randomness reproducible.

The analysis itself is based on historical market data downloaded from Yahoo Finance.

In [3]:
np.random.seed(42)
plt.rcParams["figure.figsize"] = (11, 5)

## 2. Data Collection

### S&P 500 Historical Data

The S&P 500 Index (`^GSPC`) is used as the market benchmark.

Daily closing-price data is downloaded from Yahoo Finance for the period:

- **Start:** January 1, 2015
- **End:** December 31, 2024
- **Frequency:** Daily

The raw price series is used to calculate daily log returns for the subsequent analysis.

The data is downloaded programmatically so that the analysis can be reproduced without storing the raw dataset in the repository.

In [4]:
import yfinance as yf

data = yf.download('^GSPC', start='2015-01-01', end='2024-12-31')

# yfinance >= 0.2 returns MultiIndex columns (Price, Ticker) even for a
# single ticker -- data['Close'] can come back as a 1-column DataFrame
# instead of a Series. Normalize to a plain Series either way.
if isinstance(data.columns, pd.MultiIndex):
    prices = data['Close'].iloc[:, 0]
else:
    prices = data['Close']


/tmp/ipykernel_2147/75882113.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download('^GSPC', start='2015-01-01', end='2024-12-31')
[*********************100%***********************]  1 of 1 completed


### 2.1 Log Return Calculation

Financial volatility models are generally applied to **returns** rather than price levels.

Daily log returns are calculated as:

\[
r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
\]

where:

- \(P_t\) is the closing price on day \(t\)
- \(P_{t-1}\) is the closing price on the previous trading day
- \(r_t\) is the daily log return

The first observation is removed because there is no previous price from which to calculate a return.

In [5]:
# Calculate log returns
returns = np.log(prices / prices.shift(1))

# Remove first NaN
returns = returns.dropna()

## 3. Statistical Diagnostics

Before fitting volatility models, the return series is examined to determine whether it satisfies the properties required for the subsequent time-series analysis.

Two questions are investigated:

1. **Are the returns stationary?**
2. **Is there evidence of volatility clustering?**

---

### 3.1 Augmented Dickey-Fuller (ADF) Test

The Augmented Dickey-Fuller test is used to test whether the return series contains a unit root.

#### Hypotheses

**Null hypothesis \(H_0\):** The return series has a unit root and is non-stationary.

**Alternative hypothesis \(H_1\):** The return series is stationary.

Using a significance level of 5%:

- If **p-value < 0.05**, reject \(H_0\) and conclude that the series is stationary.
- If **p-value ≥ 0.05**, do not reject \(H_0\).

The ADF test is applied to the daily S&P 500 log returns.

In [6]:
adf_stat, adf_p, *_ = adfuller(returns)
print(f"Augmented Dickey-Fuller test on returns:")
print(f"  ADF statistic = {adf_stat:.3f}, p-value = {adf_p:.30f}")
print(f"  -> {'Stationary (reject unit root)' if adf_p < 0.05 else 'NOT stationary'}")

Augmented Dickey-Fuller test on returns:
  ADF statistic = -15.730, p-value = 0.000000000000000000000000000129
  -> Stationary (reject unit root)


### ADF Test Interpretation

The ADF test produces:

- **ADF statistic:** -15.730
- **p-value:** approximately \(1.29 \times 10^{-28}\)

The p-value is far below the 5% significance level.

Therefore, the null hypothesis of a unit root is rejected.

**Conclusion:** The S&P 500 daily return series is stationary over the sample period.

This allows the analysis to proceed to volatility diagnostics and GARCH-family modelling.

### 3.2 Volatility Clustering and ARCH Effects

Financial returns often exhibit **volatility clustering**, where periods of relatively large price movements tend to be followed by other large movements, while periods of small movements tend to be followed by small movements.

To investigate this, the squared returns are analyzed.

\[
r_t^2
\]

Squaring the returns removes their sign and focuses on the **magnitude of price movements**.

If squared returns are significantly autocorrelated, this provides evidence that the magnitude of current returns is related to recent return magnitudes, which is consistent with volatility clustering and motivates the use of ARCH/GARCH-type models.

Two diagnostics are used:

1. **Ljung-Box test** on squared returns
2. **Autocorrelation Function (ACF)** of squared returns

In [7]:
# Ljung-Box on SQUARED returns: tests whether squared returns are
# autocorrelated. If they are, that IS volatility clustering, formally.
sq_returns = returns**2
lb = acorr_ljungbox(sq_returns, lags=[10, 20], return_df=True)
print("\nLjung-Box test on squared returns (tests for ARCH effects):")
print(lb)
print("-> Very low p-values mean squared returns are strongly autocorrelated,")
print("   i.e. today's volatility is predictable from recent volatility.")


Ljung-Box test on squared returns (tests for ARCH effects):
        lb_stat  lb_pvalue
10  3197.020264        0.0
20  3727.642852        0.0
-> Very low p-values mean squared returns are strongly autocorrelated,
   i.e. today's volatility is predictable from recent volatility.


### Ljung-Box Test Interpretation

The Ljung-Box test is applied to the squared return series at lags 10 and 20.

#### Hypotheses

**Null hypothesis \(H_0\):** There is no autocorrelation in the squared returns up to the specified lag.

**Alternative hypothesis \(H_1\):** There is significant autocorrelation in the squared returns.

The resulting p-values are effectively zero at both lag 10 and lag 20.

Since the p-values are far below 0.05, the null hypothesis is rejected.

**Conclusion:** The squared returns exhibit statistically significant serial dependence, providing strong evidence of volatility clustering.

This result provides motivation for fitting ARCH/GARCH-family volatility models.

### Interpretation

The Ljung-Box test is applied to squared returns to detect serial
dependence in the magnitude of returns.

The extremely small p-values indicate significant autocorrelation
in squared returns, providing evidence of volatility clustering.

This motivates the use of ARCH/GARCH-family volatility models.

### 3.3 ACF of Squared Returns

The autocorrelation function (ACF) provides a visual representation of the relationship between squared returns and their previous values.

Large ACF values that extend beyond the approximate 95% confidence bounds indicate statistically significant autocorrelation.

The ACF plot provides a visual complement to the Ljung-Box test and helps illustrate the persistence in the squared-return series.

In [8]:
# ACF of squared returns, visual confirmation
fig, ax = plt.subplots(figsize=(11, 4))
sq_acf = acf(sq_returns, nlags=40)
ax.stem(range(len(sq_acf)), sq_acf)
n = len(sq_returns)
ax.axhline(1.96 / np.sqrt(n), color="red", linestyle="--", linewidth=0.8)
ax.axhline(-1.96 / np.sqrt(n), color="red", linestyle="--", linewidth=0.8)
ax.set_title("ACF of Squared Returns (evidence of volatility clustering)")
plt.tight_layout()
plt.savefig("step2_sq_returns_acf.png", dpi=110)
plt.close()
print("Saved plot: step2_sq_returns_acf.png")

Saved plot: step2_sq_returns_acf.png


### ACF Interpretation

The ACF of squared returns provides visual evidence of serial dependence in the magnitude of returns.

Together with the Ljung-Box results, this supports the presence of volatility clustering in the S&P 500 return series.

Therefore, a conditional volatility model is appropriate for the next stage of the analysis.

## 4. GARCH-Family Volatility Models

The statistical diagnostics indicate that the return series is stationary and that squared returns exhibit significant serial dependence.

The next step is to model the **conditional volatility** of returns.

Three volatility specifications are estimated and compared:

### GARCH(1,1)

The standard GARCH model represents conditional variance as a function of past squared shocks and past conditional variance.

\[
\sigma_t^2
=
\omega
+
\alpha\epsilon_{t-1}^2
+
\beta\sigma_{t-1}^2
\]

### EGARCH(1,1)

EGARCH models volatility in logarithmic form and allows asymmetric responses to positive and negative shocks.

### GJR-GARCH(1,1)

GJR-GARCH introduces an additional asymmetric term to allow negative shocks to have a different effect on future volatility than positive shocks.

All three models are estimated using a Student-t innovation distribution to account for the heavy-tailed nature of financial returns.

The models are fitted to returns expressed in percentage points for numerical stability.

In [9]:
# arch_model works best with returns scaled to percentage points
r_pct = returns * 100

### 4.1 Model Estimation

Each candidate model is fitted to the full return series.

For every fitted model, the following quantities are recorded:

- Log-likelihood
- Akaike Information Criterion (AIC)
- Bayesian Information Criterion (BIC)

These metrics allow the competing volatility specifications to be compared using both model fit and model complexity.

In [10]:
models = {
    "GARCH(1,1)":    arch_model(r_pct, vol="GARCH", p=1, q=1, dist="t"),
    "EGARCH(1,1)":   arch_model(r_pct, vol="EGARCH", p=1, q=1, dist="t"),
    "GJR-GARCH(1,1)": arch_model(r_pct, vol="GARCH", p=1, o=1, q=1, dist="t"),
}

### 4.1 Model Estimation

Each candidate model is fitted to the full return series.

For every fitted model, the following quantities are recorded:

- Log-likelihood
- Akaike Information Criterion (AIC)
- Bayesian Information Criterion (BIC)

These metrics allow the competing volatility specifications to be compared using both model fit and model complexity.

In [11]:
results = {}
comparison_rows = []
for name, spec in models.items():
    fit = spec.fit(disp="off")
    results[name] = fit
    comparison_rows.append({
        "Model": name,
        "LogLik": fit.loglikelihood,
        "AIC": fit.aic,
        "BIC": fit.bic,
    })
    print(f"\n--- {name} ---")
    print(fit.summary().tables[2])  # volatility model parameters


--- GARCH(1,1) ---
                              Volatility Model                              
                 coef    std err          t      P>|t|      95.0% Conf. Int.
----------------------------------------------------------------------------
omega          0.0199  5.835e-03      3.415  6.382e-04 [8.489e-03,3.136e-02]
alpha[1]       0.1705  2.394e-02      7.123  1.059e-12     [  0.124,  0.217]
beta[1]        0.8264  2.164e-02     38.196      0.000     [  0.784,  0.869]

--- EGARCH(1,1) ---
                               Volatility Model                              
                 coef    std err          t      P>|t|       95.0% Conf. Int.
-----------------------------------------------------------------------------
omega          0.0105  7.167e-03      1.465      0.143 [-3.547e-03,2.455e-02]
alpha[1]       0.3186  3.543e-02      8.993  2.412e-19      [  0.249,  0.388]
beta[1]        0.9708  7.394e-03    131.293      0.000      [  0.956,  0.985]

--- GJR-GARCH(1,1) ---
     

In [12]:
comp_df = pd.DataFrame(comparison_rows).sort_values("BIC")
print("\n" + "=" * 70)
print("MODEL COMPARISON (lower AIC/BIC = better)")
print("=" * 70)
print(comp_df.to_string(index=False))

best_model_name = comp_df.iloc[0]["Model"]
best_fit = results[best_model_name]
print(f"\nBest model by BIC: {best_model_name}")


MODEL COMPARISON (lower AIC/BIC = better)
         Model       LogLik         AIC         BIC
GJR-GARCH(1,1) -3110.899462 6233.798923 6268.776706
    GARCH(1,1) -3147.905626 6305.811253 6334.959405
   EGARCH(1,1) -3154.323689 6318.647378 6347.795530

Best model by BIC: GJR-GARCH(1,1)


### Model Comparison Results

| Model | AIC | BIC |
|---|---:|---:|
| GARCH(1,1) | 6305.81 | 6334.96 |
| EGARCH(1,1) | 6318.65 | 6347.80 |
| **GJR-GARCH(1,1)** | **6233.80** | **6268.78** |

Among the three candidate models, **GJR-GARCH(1,1)** has the lowest AIC and BIC.

Therefore:

> **GJR-GARCH(1,1) is selected as the preferred model based on BIC.**

The GJR-GARCH specification also includes an asymmetric volatility term, allowing the model to capture potentially different volatility responses to negative and positive shocks.

## 5. Fitted Conditional Volatility

After selecting the preferred GARCH-family model, its estimated conditional volatility is extracted.

Conditional volatility represents the model's estimate of the time-varying standard deviation of returns.

The fitted volatility is plotted alongside the realized daily returns to visually examine whether periods of larger returns coincide with higher estimated volatility.

In [13]:
cond_vol = best_fit.conditional_volatility / 100  # back to decimal scale

In [14]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(returns.index, returns, color="lightgray", linewidth=0.6, label="Daily return")
ax.plot(returns.index, cond_vol, color="crimson", label=f"{best_model_name} conditional volatility")
ax.plot(returns.index, -cond_vol, color="crimson")
ax.set_title(f"Fitted Conditional Volatility ({best_model_name}) vs Returns")
ax.legend()
plt.tight_layout()
plt.savefig("step4_conditional_vol.png", dpi=110)
plt.close()
print("Saved plot: step4_conditional_vol.png")


Saved plot: step4_conditional_vol.png


### Interpretation

The conditional volatility estimate varies over time rather than remaining constant.

Periods associated with larger absolute returns are generally accompanied by higher estimated conditional volatility.

This time-varying behavior is the central feature being captured by the GARCH-family model.

## 6. Walk-Forward Value-at-Risk Backtesting

### 6.1 Why Value-at-Risk?

Value-at-Risk (VaR) provides a threshold for potential losses at a specified confidence level.

For a **99% one-sided VaR**, the model estimates a return threshold such that losses beyond this threshold should occur approximately 1% of the time under the model assumptions.

The analysis uses:

- **Confidence level:** 99%
- **Backtesting window:** 250 trading days
- **Forecast horizon:** 1 trading day

Rather than using a single in-sample volatility estimate, a walk-forward procedure is used to generate out-of-sample volatility forecasts.

In [15]:
test_window = 250
confidence = 0.99
# Student-t quantile at the fitted degrees of freedom, for a one-sided VaR
from scipy.stats import t as t_dist
nu = best_fit.params.get("nu", 8)  # fitted t-distribution dof
t_quantile = t_dist.ppf(1 - confidence, df=nu) * np.sqrt((nu - 2) / nu)  # standardized

### 6.2 Walk-Forward Forecasting

Walk-forward forecasting simulates how a risk model would operate in practice.

For each day in the final 250-day test period:

1. Use all observations available before that day as the training set.
2. Fit the volatility model to the training data.
3. Generate a one-step-ahead volatility forecast.
4. Calculate the corresponding VaR threshold.
5. Compare the predicted VaR threshold with the actual return observed on that day.

The process is repeated for every day in the test window.

This produces a sequence of genuinely out-of-sample VaR forecasts.

In [16]:
def create_model(train, model_name):

    if model_name == "GARCH(1,1)":
        return arch_model(
            train,
            vol="GARCH",
            p=1,
            q=1,
            dist="t"
        )

    elif model_name == "EGARCH(1,1)":
        return arch_model(
            train,
            vol="EGARCH",
            p=1,
            q=1,
            dist="t"
        )

    elif model_name == "GJR-GARCH(1,1)":
        return arch_model(
            train,
            vol="GARCH",
            p=1,
            o=1,
            q=1,
            dist="t"
        )

### 6.3 99% VaR Estimation

The volatility forecast is combined with the fitted Student-t innovation distribution to obtain the lower-tail quantile.

The resulting VaR is a **negative return threshold**.

For example, if:

\[
VaR_t = -0.025
\]

then a realized return below -2.5% would constitute a VaR breach.

The model therefore generates a new risk threshold for each day of the backtesting period.

In [17]:
forecast_vol = []
actual_ret = []

for i in range(n - test_window, n):
    train = r_pct.iloc[:i]
    spec = create_model(train, best_model_name)
    f = spec.fit(disp="off")
    fc = f.forecast(horizon=1, reindex=False)
    sigma_next = np.sqrt(fc.variance.values[-1, 0]) / 100  # back to decimal
    forecast_vol.append(sigma_next)
    actual_ret.append(returns.iloc[i])

In [18]:
forecast_vol = np.array(forecast_vol)
actual_ret = np.array(actual_ret)
mu_forecast = returns.iloc[:n - test_window].mean()  # simple constant mean assumption
VaR = mu_forecast + t_quantile * forecast_vol  # this is a NEGATIVE number (a loss threshold)

### 6.4 VaR Breach Analysis

A **VaR breach** occurs when the realized return is below the forecast VaR threshold:

\[
r_t < VaR_t
\]

At a 99% confidence level, the expected breach probability is:

\[
1 - 0.99 = 0.01
\]

Therefore, over 250 trading days, the expected number of breaches is approximately:

\[
250 \times 0.01 = 2.5
\]

The observed number of breaches is compared with this expected frequency.

In [19]:
breaches = actual_ret < VaR
n_breaches = breaches.sum()
expected_breaches = (1 - confidence) * test_window

print(f"Test window: last {test_window} trading days")
print(f"Expected breaches at {int(confidence*100)}% VaR: ~{expected_breaches:.1f}")
print(f"Actual breaches observed: {n_breaches}")
print(f"Breach rate: {n_breaches/test_window:.2%} (target: {1-confidence:.2%})")

Test window: last 250 trading days
Expected breaches at 99% VaR: ~2.5
Actual breaches observed: 5
Breach rate: 2.00% (target: 1.00%)


### VaR Backtest Results

The 250-day backtesting period produces:

- **Expected breaches:** approximately 2.5
- **Observed breaches:** 5
- **Observed breach rate:** 2.00%
- **Target breach rate:** 1.00%

The observed breach rate is higher than the theoretical 1% target.

However, the difference in breach frequency must be formally tested rather than judged from the breach count alone.

The Kupiec unconditional coverage test is therefore applied next.

## 7. Kupiec Unconditional Coverage Test

The Kupiec test evaluates whether the observed frequency of VaR breaches is statistically consistent with the target VaR level.

For a 99% VaR:

\[
p = 0.01
\]

### Hypotheses

**Null hypothesis \(H_0\):**

The true probability of a VaR breach is equal to the expected probability of 1%.

**Alternative hypothesis \(H_1\):**

The true breach probability differs from 1%.

Using a 5% significance level:

- **p-value > 0.05:** Fail to reject \(H_0\)
- **p-value < 0.05:** Reject \(H_0\)

The test therefore determines whether the observed number of VaR breaches is statistically inconsistent with the 99% VaR target.

In [20]:
# Kupiec unconditional coverage test (standard VaR backtest in risk teams)
pi_hat = n_breaches / test_window
if 0 < pi_hat < 1:
    LR_uc = -2 * (
        (test_window - n_breaches) * np.log(1 - (1 - confidence)) + n_breaches * np.log(1 - confidence)
        - ((test_window - n_breaches) * np.log(1 - pi_hat) + n_breaches * np.log(pi_hat))
    )
    from scipy.stats import chi2
    kupiec_p = 1 - chi2.cdf(LR_uc, df=1)
    print(f"\nKupiec test statistic: {LR_uc:.3f}, p-value: {kupiec_p:.3f}")
    print("-> " + ("PASS: breach rate consistent with 99% VaR (fail to reject)"
                    if kupiec_p > 0.05 else
                    "FAIL: breach rate significantly off from 99% VaR target"))


Kupiec test statistic: 1.957, p-value: 0.162
-> PASS: breach rate consistent with 99% VaR (fail to reject)


### Kupiec Test Interpretation

The Kupiec test produces:

- **Test statistic:** 1.957
- **p-value:** 0.162

Since:

\[
0.162 > 0.05
\]

the null hypothesis is not rejected.

**Conclusion:** The observed VaR breach frequency is not statistically inconsistent with the 99% target at the 5% significance level.

Therefore, the VaR forecasts **pass the Kupiec unconditional coverage test** for this backtesting window.

It is important to note that passing the Kupiec test only evaluates unconditional breach frequency. It does not test whether breaches are independent over time.

## 8. Visual VaR Backtest

The final visualization compares:

- Realized daily returns
- Forecast 99% VaR threshold
- Days on which the realized return breached the VaR threshold

A breach occurs when the realized return falls below the forecast VaR line.

This provides an intuitive visual assessment of how the estimated risk threshold behaved during the final 250 trading days.

In [21]:
fig, ax = plt.subplots(figsize=(11, 5))
test_dates = returns.index[-test_window:]
ax.plot(test_dates, actual_ret, color="steelblue", linewidth=0.9, label="Actual return")
ax.plot(test_dates, VaR, color="darkred", linewidth=1.2, label="99% VaR threshold (forecast)")
ax.scatter(test_dates[breaches], actual_ret[breaches], color="red", zorder=5, label="Breach")
ax.set_title("Walk-Forward 99% VaR Backtest - Last 250 Trading Days")
ax.legend()
plt.tight_layout()
plt.savefig("step5_var_backtest.png", dpi=110)
plt.close()
print("Saved plot: step5_var_backtest.png")


Saved plot: step5_var_backtest.png


# 9. Final Conclusions

This project developed an end-to-end volatility forecasting and market risk pipeline for the S&P 500.

### Main Findings

**1. Returns are stationary**

The ADF test strongly rejects the unit-root hypothesis, indicating that the daily return series is stationary over the sample period.

**2. Volatility clustering is present**

The Ljung-Box test on squared returns produces extremely small p-values, providing strong evidence of serial dependence in squared returns and volatility clustering.

**3. GJR-GARCH performs best among the tested models**

GARCH(1,1), EGARCH(1,1), and GJR-GARCH(1,1) were compared using AIC and BIC.

GJR-GARCH(1,1) achieves the lowest BIC and is therefore selected as the preferred specification.

**4. Volatility is time-varying**

The fitted conditional volatility changes over time, capturing periods of relatively high and low market uncertainty.

**5. VaR forecasts were evaluated out-of-sample**

A walk-forward procedure was used to generate 99% one-day-ahead VaR forecasts over the final 250 trading days.

**6. VaR passes the Kupiec coverage test**

Five breaches were observed compared with approximately 2.5 expected breaches.

Although the observed breach rate was 2.00%, the Kupiec test produced a p-value of 0.162. Therefore, the null hypothesis of the target 1% breach probability was not rejected at the 5% significance level.

### Overall

The analysis demonstrates how volatility diagnostics, GARCH-family modelling, out-of-sample forecasting, and VaR backtesting can be combined into a complete market-risk modelling workflow.

## 10. Limitations and Potential Extensions

The current analysis provides a focused implementation of GARCH-family volatility forecasting and VaR backtesting, but several extensions are possible.

### Limitations

- The analysis focuses on a single market index, the S&P 500.
- The VaR backtest uses a 250-trading-day evaluation window.
- The Kupiec test evaluates unconditional coverage but does not test the independence of VaR breaches.
- Model selection is based on information criteria rather than solely on out-of-sample forecasting performance.

### Potential Extensions

Future work could include:

- Christoffersen conditional coverage testing
- Comparison of additional volatility specifications
- Alternative VaR confidence levels such as 95% and 99%
- Expected Shortfall (ES)
- Longer rolling and expanding-window backtests
- Comparison across multiple equity indices or assets
- Forecast accuracy evaluation using out-of-sample volatility measures